## Fine-Tuning Neural Network Hyperparameters with Optuna

In [12]:
import torch
import torch.nn as nn
import torchvision 
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor
)

test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000]
)

In [13]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [14]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )
    
    def forward(self, X):
        return self.mlp(X)

In [15]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

Let’s tune the learning rate and the number of neurons in the hidden layers (for simplicity, we will use the same number of neurons in both hidden layers). First, we need to define a function that Optuna will call many times to perform hyperparameter tuning: this function
must take a Trial object and use it to ask Optuna for hyperparameter values, and then use these hyperparameter values to build and train a model. Finally, the function must evaluate the model (typically on the validation set) and return the metric.

In [16]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            preds = torch.argmax(y_pred, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

def objective(trail):
    learning_rate = trail.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trail.suggest_int("n_hidden", 20, 300)

    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, 
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    n_epochs = 20

    train(model, optimizer, xentropy, train_loader, n_epochs)
    valid_acc = evaluate(model, valid_loader)

    return valid_acc

In [ ]:
import optuna

torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=3)

d:\programming\machine_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-03 19:00:07,356] A new study created in memory with name: no-name-c5dc5e60-60c3-4f56-8f61-3026038caf0b


Epoch 1/20, Loss: 2.2769
Epoch 2/20, Loss: 2.2093
Epoch 3/20, Loss: 2.1164
Epoch 4/20, Loss: 1.9776
Epoch 5/20, Loss: 1.7867
Epoch 6/20, Loss: 1.5775
Epoch 7/20, Loss: 1.3979
Epoch 8/20, Loss: 1.2605
Epoch 9/20, Loss: 1.1573
Epoch 10/20, Loss: 1.0782
Epoch 11/20, Loss: 1.0162
Epoch 12/20, Loss: 0.9665
Epoch 13/20, Loss: 0.9258
Epoch 14/20, Loss: 0.8918
Epoch 15/20, Loss: 0.8629
Epoch 16/20, Loss: 0.8382
Epoch 17/20, Loss: 0.8165
Epoch 18/20, Loss: 0.7974
Epoch 19/20, Loss: 0.7803
Epoch 20/20, Loss: 0.7649


[I 2026-07-03 19:10:02,586] Trial 0 finished with value: 0.7102 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7102.


Epoch 1/20, Loss: 1.1392
Epoch 2/20, Loss: 0.6219
Epoch 3/20, Loss: 0.5251
Epoch 4/20, Loss: 0.4826
Epoch 5/20, Loss: 0.4571
Epoch 6/20, Loss: 0.4405
Epoch 7/20, Loss: 0.4239


In [ ]:
print(study.best_params)
print(study.best_value)

Optuna comes with several Pruner classes that can detect and prune bad trials. For example, the MedianPruner will prune trials whose performance is below the median performance, at regular intervals during training. 

In [ ]:
pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3, interval_steps=1)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

Now we need a slightly modified objective function in which after each epoch, it checks validation accuracy and reports it to optuna of the current validation accuracy and epoch so it can determine whether the trail should be pruned.

In [ ]:
def objective(trail):
    learning_rate = trail.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trail.suggest_int("n_hidden", 20, 300)

    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, 
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    n_epochs = 20

    # instead of the train function call, we paste the modified train function
     
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

        #added part for pruning 
        
        valid_acc = evaluate(model, valid_loader)
        trail.report(valid_acc, epoch)
        if trail.should_prune():
            raise optuna.TrailPruned()

    return valid_acc